In [1]:
import os
import time
import random
import logging
import warnings
import openpyxl
from datetime import datetime, timedelta, date
from pathlib import Path

import pandas as pd
import numpy as np
from pytrends.request import TrendReq
from tqdm import tqdm

warnings.filterwarnings("ignore")

In [2]:
KEYWORDS = [
    'diabetes', 'sering haus', 'sering buang air kecil', 'badan lemas',
    'penglihatan kabur', 'luka sulit sembuh', 'gula darah tinggi',
    'cek gula darah', 'diet diabetes', 'obesitas',
    'minuman manis', 'makanan manis', 'neuropati', 'retinopati',
    'gagal ginjal', 'kaki diabetes', 'insulin', 'prediabetes',
    'hiperglikemia', 'kadar gula normal'
]

CONFIG = {
    "geo"              : "ID",           # Indonesia
    "hl"               : "id-ID",        # Bahasa Indonesia
    "tz"               : 420,            # WIB (UTC+7)
    "start_date"       : "2015-01-01",   # 10 tahun ke belakang dari 2025
    "end_date"         : datetime.today().strftime('%Y-%m-%d'),   # Hari ini
    "chunk_months"     : 6,              # Ukuran potongan timeframe (< 9 bulan)
    "keywords_per_req" : 5,              # Max keyword per request pytrends
    "sleep_min"        : 3.0,            # Jeda minimal antar request (detik)
    "sleep_max"        : 7.0,            # Jeda maksimal antar request (detik)
    "max_retries"      : 5,              # Percobaan ulang jika gagal
    "retry_wait"       : 60,             # Tunggu (detik) jika kena rate limit
    "output_dir"       : "output_trends", # Folder output
    "save_checkpoint"  : True,           # Simpan progress per chunk
    "cat"              : 0,              # Kategori: 0 = semua kategori
}

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP LOGGING
# ─────────────────────────────────────────────────────────────────────────────

Path(CONFIG["output_dir"]).mkdir(exist_ok=True)
log_path = os.path.join(CONFIG["output_dir"], "scraper.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_path, encoding="utf-8"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def generate_date_chunks(start: str, end: str, chunk_months: int) -> list:
    """
    Memecah rentang tanggal menjadi chunk-chunk kecil.
    Setiap chunk < 9 bulan agar Google Trends mengembalikan data HARIAN.
    """
    chunks = []
    current = datetime.strptime(start, "%Y-%m-%d")
    end_dt  = datetime.strptime(end, "%Y-%m-%d")

    while current < end_dt:
        next_month = current.month + chunk_months
        next_year  = current.year + (next_month - 1) // 12
        next_month = (next_month - 1) % 12 + 1
        chunk_end  = datetime(next_year, next_month, 1) - timedelta(days=1)
        chunk_end  = min(chunk_end, end_dt)

        chunks.append((
            current.strftime("%Y-%m-%d"),
            chunk_end.strftime("%Y-%m-%d")
        ))
        current = chunk_end + timedelta(days=1)

    log.info(f"Total chunks: {len(chunks)}")
    return chunks


def batch_keywords(keywords: list, batch_size: int) -> list:
    """Memecah list keyword menjadi batch-batch kecil."""
    return [keywords[i:i+batch_size] for i in range(0, len(keywords), batch_size)]


def build_pytrends() -> TrendReq:
    """Inisialisasi pytrends dengan konfigurasi Indonesia."""
    return TrendReq(
        hl=CONFIG["hl"],
        tz=CONFIG["tz"],
        timeout=(10, 30),
        retries=2,
        backoff_factor=0.5
    )


def fetch_chunk(pytrends: TrendReq, keywords: list, start: str, end: str) -> pd.DataFrame:
    """
    Fetch data trends untuk satu chunk waktu + satu batch keyword.
    Mengembalikan DataFrame dengan kolom = keyword, index = tanggal.
    """
    timeframe = f"{start} {end}"
    for attempt in range(1, CONFIG["max_retries"] + 1):
        try:
            pytrends.build_payload(
                keywords,
                cat=CONFIG["cat"],
                timeframe=timeframe,
                geo=CONFIG["geo"],
                gprop=""
            )
            df = pytrends.interest_over_time()

            if df.empty:
                log.warning(f"  Data kosong: {keywords} | {timeframe}")
                return pd.DataFrame()

            if "isPartial" in df.columns:
                df.drop(columns=["isPartial"], inplace=True)

            return df

        except Exception as e:
            err_msg = str(e)
            log.warning(f"  Attempt {attempt}/{CONFIG['max_retries']} gagal: {err_msg[:120]}")

            if "429" in err_msg or "Too Many Requests" in err_msg:
                wait = CONFIG["retry_wait"] * attempt
                log.info(f"  Rate limited! Tunggu {wait}s ...")
                time.sleep(wait)
            else:
                time.sleep(CONFIG["sleep_max"] * attempt)

            pytrends = build_pytrends()

    log.error(f"  Gagal setelah {CONFIG['max_retries']} percobaan: {keywords} | {timeframe}")
    return pd.DataFrame()


def normalize_and_merge(chunk_dfs: list) -> pd.DataFrame:
    if not chunk_dfs:
        return pd.DataFrame()

    merged = pd.concat(chunk_dfs, axis=0)
    merged = merged[~merged.index.duplicated(keep='last')]
    merged.sort_index(inplace=True)
    return merged

MASTER_PATH = os.path.join(CONFIG["output_dir"], "diabetes_data.csv")

def update_master_data(new_df):
    if os.path.exists(MASTER_PATH):
        old_df = pd.read_csv(MASTER_PATH, index_col=0, parse_dates=True)
        final_df = pd.concat([old_df, new_df])
        final_df = final_df[~final_df.index.duplicated(keep='last')]
    else:
        final_df = new_df

    final_df.sort_index(inplace=True)
    final_df.to_csv(MASTER_PATH)

    return final_df

def save_checkpoint(df: pd.DataFrame, keyword_batch_idx: int, chunk_idx: int):
    """Simpan checkpoint ke CSV untuk recovery jika scraper terhenti."""
    chk_dir = os.path.join(CONFIG["output_dir"], "checkpoints")
    Path(chk_dir).mkdir(exist_ok=True)
    fname = os.path.join(chk_dir, f"batch{keyword_batch_idx}_chunk{chunk_idx}.csv")
    df.to_csv(fname)
    log.debug(f"  Checkpoint disimpan: {fname}")


def load_existing_results() -> dict:
    """Load hasil yang sudah ada dari checkpoint sebelumnya."""
    chk_dir = os.path.join(CONFIG["output_dir"], "checkpoints")
    if not os.path.exists(chk_dir):
        return {}

    results = {}
    for f in Path(chk_dir).glob("*.csv"):
        parts = f.stem.split("_")
        try:
            b_idx = int(parts[0].replace("batch", ""))
            c_idx = int(parts[1].replace("chunk", ""))
            df = pd.read_csv(f, index_col=0, parse_dates=True)
            results[(b_idx, c_idx)] = df
            log.info(f"  Checkpoint dimuat: {f.name}")
        except Exception:
            pass
    return results

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# MAIN SCRAPER
# ─────────────────────────────────────────────────────────────────────────────


def scrape_trends() -> pd.DataFrame:
    
    log.info("=" * 70)
    log.info("MULAI SCRAPING GOOGLE TRENDS - DETEKSI DIABETES INDONESIA")
    log.info(f"Keywords  : {len(KEYWORDS)} keyword")
    log.info(f"Periode   : {CONFIG['start_date']} s/d {CONFIG['end_date']}")
    log.info(f"Granulari : Harian (via chunk {CONFIG['chunk_months']} bulan)")
    log.info(f"Geo       : {CONFIG['geo']}")
    log.info("=" * 70)

    date_chunks    = generate_date_chunks(CONFIG["start_date"], CONFIG["end_date"], CONFIG["chunk_months"])
    keyword_batches = batch_keywords(KEYWORDS, CONFIG["keywords_per_req"])

    log.info(f"Chunks waktu   : {len(date_chunks)}")
    log.info(f"Batch keyword  : {len(keyword_batches)} (@ max 5 keyword)")
    total_requests = len(date_chunks) * len(keyword_batches)
    log.info(f"Total requests : {total_requests}")
    est_time = total_requests * ((CONFIG['sleep_min'] + CONFIG['sleep_max']) / 2)
    log.info(f"Estimasi waktu : ~{est_time/60:.1f} menit (termasuk sleep)")
    log.info("-" * 70)

    existing = load_existing_results()
    all_batch_dfs = {i: [] for i in range(len(keyword_batches))}

    pytrends = build_pytrends()

    pbar = tqdm(total=total_requests, desc="Scraping Progress", unit="req")

    for b_idx, kw_batch in enumerate(keyword_batches):
        log.info(f"\n[Batch {b_idx+1}/{len(keyword_batches)}] Keywords: {kw_batch}")

        for c_idx, (start, end) in enumerate(date_chunks):
            key = (b_idx, c_idx)

            if key in existing:
                all_batch_dfs[b_idx].append(existing[key])
                pbar.update(1)
                continue

            log.info(f"  Chunk [{c_idx+1}/{len(date_chunks)}]: {start} ~ {end}")
            df = fetch_chunk(pytrends, kw_batch, start, end)

            if not df.empty:
                all_batch_dfs[b_idx].append(df)
                if CONFIG["save_checkpoint"]:
                    save_checkpoint(df, b_idx, c_idx)

            sleep_time = random.uniform(CONFIG["sleep_min"], CONFIG["sleep_max"])
            time.sleep(sleep_time)
            pbar.update(1)

    pbar.close()

    log.info("\nMenggabungkan semua hasil ...")
    final_dfs = []
    for b_idx, dfs in all_batch_dfs.items():
        if dfs:
            batch_merged = normalize_and_merge(dfs)
            final_dfs.append(batch_merged)

    if not final_dfs:
        log.error("Tidak ada data yang berhasil diambil!")
        return pd.DataFrame()

    final_df = pd.concat(final_dfs, axis=1)
    final_df = final_df[~final_df.index.duplicated(keep='last')]
    final_df.sort_index(inplace=True)

    for kw in KEYWORDS:
        if kw not in final_df.columns:
            final_df[kw] = np.nan

    final_df = final_df[KEYWORDS]  

    final_df = update_master_data(final_df)
    save_results(final_df)
    return final_df

def get_last_date():
    try:
        df = load_latest_result()
        return df.index.max().strftime("%Y-%m-%d")
    except:
        return CONFIG["start_date"]

def save_results(df: pd.DataFrame):
    """Simpan hasil ke berbagai format output."""
    output_dir = CONFIG["output_dir"]
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    # 1. CSV utama
    csv_path = os.path.join(output_dir, f"diabetes_trends_10yr_{ts}.csv")
    old_df = load_latest_result()
    new_df = df

    final_df = pd.concat([old_df, new_df])
    final_df = final_df[~final_df.index.duplicated(keep='last')]
    final_df.to_csv("output_trends/master_data.csv")
    log.info(f"\n✅ CSV disimpan     : {csv_path}")

    # 2. Excel dengan multiple sheet
    xlsx_path = os.path.join(output_dir, f"diabetes_trends_10yr_{ts}.xlsx")
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="Raw Data")

        # Sheet statistik
        stats = df.describe().T
        stats["missing_%"] = (df.isnull().sum() / len(df) * 100).values
        stats.to_excel(writer, sheet_name="Statistik")

        # Sheet per tahun (agregasi tahunan)
        yearly = df.resample("YE").mean().round(2)
        yearly.index = yearly.index.year
        yearly.index.name = "Tahun"
        yearly.to_excel(writer, sheet_name="Rata-rata Tahunan")

        # Sheet per bulan
        monthly = df.resample("ME").mean().round(2)
        monthly.to_excel(writer, sheet_name="Rata-rata Bulanan")

    log.info(f"✅ Excel disimpan   : {xlsx_path}")

    # 3. Laporan ringkasan
    report_path = os.path.join(output_dir, f"laporan_ringkasan_{ts}.txt")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write("LAPORAN RINGKASAN SCRAPING GOOGLE TRENDS\n")
        f.write("Proyek: Deteksi Diabetes via Big Data\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Tanggal scraping : {datetime.now().strftime('%d %B %Y %H:%M')}\n")
        f.write(f"Periode data     : {df.index.min().date()} s/d {df.index.max().date()}\n")
        f.write(f"Total baris      : {len(df):,} hari\n")
        f.write(f"Total keyword    : {len(df.columns)}\n\n")
        f.write("-" * 60 + "\n")
        f.write("STATISTIK PER KEYWORD (rata-rata interest):\n")
        f.write("-" * 60 + "\n")
        for kw in KEYWORDS:
            if kw in df.columns:
                mean_val = df[kw].mean()
                max_val  = df[kw].max()
                max_date = df[kw].idxmax()
                missing  = df[kw].isnull().sum()
                f.write(f"\n{kw}\n")
                f.write(f"  Rata-rata : {mean_val:.2f}\n")
                f.write(f"  Puncak    : {max_val:.0f} pada {max_date.date()}\n")
                f.write(f"  Data kosong: {missing} hari\n")

        f.write("\n" + "=" * 60 + "\n")
        f.write("TOP 5 KEYWORD TERTINGGI (rata-rata):\n")
        top5 = df.mean().sort_values(ascending=False).head(5)
        for rank, (kw, val) in enumerate(top5.items(), 1):
            f.write(f"  {rank}. {kw} : {val:.2f}\n")

    log.info(f"✅ Laporan disimpan : {report_path}")
    log.info("\n" + "=" * 70)
    log.info("SCRAPING SELESAI!")
    log.info(f"Output tersimpan di folder: ./{output_dir}/")
    log.info("=" * 70)


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# UTILITAS TAMBAHAN
# ─────────────────────────────────────────────────────────────────────────────

def load_latest_result() -> pd.DataFrame:
    """Load file CSV terbaru dari folder output."""
    output_dir = CONFIG["output_dir"]
    csvs = sorted(Path(output_dir).glob("diabetes_trends_10yr_*.csv"))
    if not csvs:
        raise FileNotFoundError("Belum ada file hasil scraping.")
    latest = csvs[-1]
    log.info(f"Loading: {latest}")
    return pd.read_csv(latest, index_col=0, parse_dates=True)


def quick_analysis(df: pd.DataFrame):
    """Analisis cepat setelah scraping selesai."""
    print("\n" + "=" * 60)
    print("ANALISIS CEPAT DATA GOOGLE TRENDS DIABETES")
    print("=" * 60)
    print(f"\nShape     : {df.shape}")
    print(f"Periode   : {df.index.min().date()} ~ {df.index.max().date()}")
    print(f"\nData sampel (5 baris pertama):")
    print(df.head())
    print(f"\nRata-rata interest per keyword (dari 0-100):")
    print(df.mean().sort_values(ascending=False).round(2).to_string())
    print(f"\nKorelasi antar keyword (Top 5 tertinggi):")
    corr = df.corr()
    pairs = []
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            pairs.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))
    top_corr = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)[:5]
    for kw1, kw2, r in top_corr:
        print(f"  {kw1} <-> {kw2} : {r:.3f}")

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("""
╔══════════════════════════════════════════════════════════════╗
║     GOOGLE TRENDS SCRAPER - BIG DATA DETEKSI DIABETES       ║
║     Keywords: 20 keyword diabetes | Periode: 10 Tahun       ║
║     Granularitas: Harian | Wilayah: Indonesia (ID)          ║
╚══════════════════════════════════════════════════════════════╝
""")

    try:
        result_df = scrape_trends()
        if not result_df.empty:
            quick_analysis(result_df)
    except KeyboardInterrupt:
        log.info("\nScraping dihentikan oleh user. Checkpoint tersimpan di output_trends/checkpoints/")
        log.info("Jalankan ulang script untuk melanjutkan dari checkpoint terakhir.")


2026-04-08 14:31:31,827 [INFO] ======================================================================
2026-04-08 14:31:31,828 [INFO] MULAI SCRAPING GOOGLE TRENDS - DETEKSI DIABETES INDONESIA
2026-04-08 14:31:31,829 [INFO] Keywords  : 20 keyword
2026-04-08 14:31:31,830 [INFO] Periode   : 2015-01-01 s/d 2026-04-08
2026-04-08 14:31:31,831 [INFO] Granulari : Harian (via chunk 6 bulan)
2026-04-08 14:31:31,832 [INFO] Geo       : ID
2026-04-08 14:31:31,832 [INFO] ======================================================================
2026-04-08 14:31:31,834 [INFO] Total chunks: 23
2026-04-08 14:31:31,835 [INFO] Chunks waktu   : 23
2026-04-08 14:31:31,835 [INFO] Batch keyword  : 4 (@ max 5 keyword)
2026-04-08 14:31:31,836 [INFO] Total requests : 92
2026-04-08 14:31:31,836 [INFO] Estimasi waktu : ~7.7 menit (termasuk sleep)
2026-04-08 14:31:31,837 [INFO] ----------------------------------------------------------------------
2026-04-08 14:31:31,843 [INFO]   Checkpoint dimuat: batch0_chunk0.csv
20


╔══════════════════════════════════════════════════════════════╗
║     GOOGLE TRENDS SCRAPER - BIG DATA DETEKSI DIABETES       ║
║     Keywords: 20 keyword diabetes | Periode: 10 Tahun       ║
║     Granularitas: Harian | Wilayah: Indonesia (ID)          ║
╚══════════════════════════════════════════════════════════════╝



2026-04-08 14:31:32,032 [INFO]   Checkpoint dimuat: batch2_chunk15.csv
2026-04-08 14:31:32,035 [INFO]   Checkpoint dimuat: batch2_chunk16.csv
2026-04-08 14:31:32,039 [INFO]   Checkpoint dimuat: batch2_chunk17.csv
2026-04-08 14:31:32,043 [INFO]   Checkpoint dimuat: batch2_chunk18.csv
2026-04-08 14:31:32,047 [INFO]   Checkpoint dimuat: batch2_chunk19.csv
2026-04-08 14:31:32,051 [INFO]   Checkpoint dimuat: batch2_chunk2.csv
2026-04-08 14:31:32,054 [INFO]   Checkpoint dimuat: batch2_chunk20.csv
2026-04-08 14:31:32,058 [INFO]   Checkpoint dimuat: batch2_chunk3.csv
2026-04-08 14:31:32,061 [INFO]   Checkpoint dimuat: batch2_chunk4.csv
2026-04-08 14:31:32,065 [INFO]   Checkpoint dimuat: batch2_chunk5.csv
2026-04-08 14:31:32,068 [INFO]   Checkpoint dimuat: batch2_chunk6.csv
2026-04-08 14:31:32,072 [INFO]   Checkpoint dimuat: batch2_chunk7.csv
2026-04-08 14:31:32,075 [INFO]   Checkpoint dimuat: batch2_chunk8.csv
2026-04-08 14:31:32,079 [INFO]   Checkpoint dimuat: batch2_chunk9.csv
2026-04-08 14:


ANALISIS CEPAT DATA GOOGLE TRENDS DIABETES

Shape     : (3750, 20)
Periode   : 2015-01-01 ~ 2025-04-07

Data sampel (5 baris pertama):
            diabetes  sering haus  sering buang air kecil  badan lemas  \
date                                                                     
2015-01-01        46            0                       8            0   
2015-01-02        47            0                       0            8   
2015-01-03        42            0                       0            0   
2015-01-04        52            0                       5            8   
2015-01-05        59            0                       0            5   

            penglihatan kabur  luka sulit sembuh  gula darah tinggi  \
date                                                                  
2015-01-01                  0                  0                  0   
2015-01-02                  0                  0                  0   
2015-01-03                  0                  0             